In [ ]:
import os
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Konfigurasi path NLTK lokal jika ada
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
if os.path.exists("nltk_data"):
    nltk.data.path.insert(0, os.path.abspath("nltk_data"))

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


## Konfigurasi

In [ ]:
# ===================== KONFIGURASI PIPELINE =====================
# Semua parameter yang sering diubah dikumpulkan di sini.
START_DATE = pd.Timestamp("2021-09-01")
END_DATE   = pd.Timestamp("2026-09-01 23:59:59")
start_date, end_date = START_DATE, END_DATE      # alias agar sel lama tetap jalan

MIN_WORDS           = 350    # panjang minimal artikel (kata)
NEAR_DUP_THRESHOLD  = 0.90   # cosine TF-IDF (judul+preview) di atas ini dianggap duplikat
NEAR_DUP_WINDOW_DAY = 3      # duplikat hanya dicari dalam jendela +-3 hari
TITLE_REPEAT        = 1      # sebelumnya 2 (judul digandakan). 1 = tidak digandakan
CUTOFF_HOUR_WIB     = 15     # berita setelah jam ini (WIB) dihitung ke hari bursa berikutnya
MIN_DOC_FREQ        = 5      # kata harus muncul di >= N artikel
MAX_DOC_FREQ_RATIO  = 0.80   # kata di > 80% artikel dibuang (template/kata hasil seleksi scraper). Jangan terlalu rendah: kata pembeda sungguhan bisa ikut terbuang

In [ ]:
news_raw = pd.read_csv('data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


In [ ]:
news_raw.info()

In [ ]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["tanggal"] = news_raw["date_raw_dt"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt"]].head(5)


In [ ]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

# Data Cleaning
## Filter Dataset

In [ ]:
# Filter rentang tanggal + hapus duplikat
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
print(f"Baris awal                         : {len(news_raw)}")

news = news[news["date_raw_dt"].between(START_DATE, END_DATE, inclusive="both")]
print(f"Setelah filter rentang {START_DATE.date()} - {END_DATE.date()} : {len(news)}")

# --- Dedup URL ---
# URL yang sama ter-scrape berulang dengan query string berbeda
# (mis. '...html?&qsearchterm=geopolitical risk'), sehingga dedup pada URL mentah tidak menangkapnya.
news["url_clean"] = (
    news["url"].fillna("").astype(str)
    .str.split(r"[?#]", n=1, regex=True).str[0]
    .str.rstrip("/").str.lower()
)

# Gabungkan keyword_matched dari semua baris duplikat agar informasi keyword tidak hilang.
kw_union = (
    news.groupby("url_clean")["keyword_matched"]
    .agg(lambda s: " | ".join(sorted({str(x).strip() for x in s.dropna() if str(x).strip()})))
)
is_dup_url = news.duplicated(subset=["url_clean"], keep="first") & news["url_clean"].ne("")
news = news.loc[~is_dup_url].copy()
news["keyword_matched"] = news["url_clean"].map(kw_union).replace("", np.nan)
print(f"Setelah dedup url (tanpa query)    : {len(news)}")

# --- Dedup judul ---
news["title_norm"] = (
    news["title"].fillna("").str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
)
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+tanggal+jam    : {len(news)}")

dup_title_day = news.duplicated(subset=["title_norm", "tanggal"], keep="first") & news["title_norm"].ne("")
news = news.loc[~dup_title_day].copy()
print(f"Setelah dedup judul (normalisasi)  : {len(news)}")

In [ ]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

In [ ]:
# Panjang text minimal MIN_WORDS kata
effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= MIN_WORDS]
print(f"Setelah filter panjang teks min {MIN_WORDS} : {len(news)}")

In [ ]:
# Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

# PERBAIKAN: pencocokan sebelumnya memakai substring (`kw in haystack`), sehingga
# 'nato' ikut cocok dengan 'senator'/'national'. Sekarang memakai batas kata (word boundary).
RELEVANCE_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in RELEVANCE_KEYWORDS) + r")(?:s)?\b",
    flags=re.IGNORECASE,
)

def is_relevant(row):
    """Relevan jika keyword_matched terisi ATAU judul/preview memuat kata kunci (word boundary)."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}"
    return bool(RELEVANCE_RE.search(haystack))

mask_relevant = news.apply(is_relevant, axis=1)
print(f"Dibuang oleh filter relevansi       : {(~mask_relevant).sum()}")
news = news.loc[mask_relevant]
print(f"Setelah filter relevansi kata kunci : {len(news)}")

# Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)

In [ ]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

## 1b. Perbaikan Teks Mentah (noise struktural)
Noise yang terlihat pada hasil sebelumnya dan ditangani di sini:
- **Kata menempel** akibat scraper, mis. `asgeopoliticalrisks`, `amidgeopoliticalturmoil`.
- **Karakter tak terlihat** (zero-width space, non-breaking space).
- **Template judul** (mis. awalan "Daily Open:" / "Morning Call Sheet:") yang membuat kata `daily` dan `open` mendominasi judul.
- **Boilerplate CNBC** di awal/akhir teks ("In this article", ajakan berlangganan, dsb.).

Setiap pola dihitung jumlah baris yang terkena supaya bisa diperiksa manual sebelum dipercaya.

In [ ]:
from collections import Counter

ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff\u00ad]")
NBSP_RE = re.compile(r"[\u00a0\u2007\u202f]")

# --- 1) Kata menempel di sekitar stem 'geopolitic*' ---
def split_glued_stems(text):
    # beri spasi hanya jika stem menempel dengan huruf lain; 'geopolitically' tidak disentuh
    text = re.sub(r"(?i)(?<=[a-z])(geopolitic(?:al(?!ly)|s))", r" \1", text)
    text = re.sub(r"(?i)(geopolitic(?:al(?!ly)|s))(?=[a-z])", r"\1 ", text)
    return text

# --- 2) Frasa kata kunci multi-kata yang menempel ('armedconflict' -> 'armed conflict') ---
def _make_squash_fix(phrase):
    rx = re.compile(re.escape(phrase.replace(" ", "")) + r"(s?)", re.IGNORECASE)
    return rx, (lambda m, p=phrase: f" {p}{m.group(1)} ")

SQUASH_FIXES = [_make_squash_fix(k) for k in RELEVANCE_KEYWORDS if " " in k]

# --- 3) Template judul ---
TITLE_TEMPLATE_RES = {
    "daily_open":       re.compile(r"(?i)^\s*(?:the\s+)?daily open\s*[:\-\u2013\u2014|]\s*"),
    "morning_call":     re.compile(r"(?i)^\s*morning call sheet\s*[:\-\u2013\u2014|]\s*"),
}

# --- 4) Boilerplate badan teks (case-sensitive untuk pola HURUF BESAR) ---
BOILERPLATE_RES = {
    "in_this_article":  re.compile(r"^\s*In this article\b"),
    "follow_stocks":    re.compile(r"Follow your favorite stocks\s+CREATE FREE ACCOUNT"),
    "watch_cta":        re.compile(r"\bWATCH(?: LIVE)?:[^.!?]{0,160}[.!?]?"),
    "read_more_cta":    re.compile(r"\b(?:READ MORE|Read more|RELATED)\s*:[^.!?]{0,160}[.!?]?"),
    "contributed_to":   re.compile(r"[\u2014\u2013-]\s*CNBC'?s?\s+[A-Z][\w.,&' -]{2,80}?contributed to this (?:report|story)\.?"),
    "subscribe_cta":    re.compile(r"(?i)\b(?:sign up for|subscribe to|don'?t miss)\b[^.!?]{0,120}\b(?:cnbc|newsletter|inbox|pro)\b[^.!?]{0,80}[.!?]?"),
    "developing_story": re.compile(r"(?i)this is a developing story\.?\s*(?:please )?check back for updates\.?"),
    "pro_cta":          re.compile(r"CNBC PRO subscribers?[^.!?]{0,160}[.!?]?"),
}

def clean_raw_text(text, is_title=False):
    if not isinstance(text, str):
        return text
    t = ZERO_WIDTH_RE.sub("", text)
    t = NBSP_RE.sub(" ", t)
    t = split_glued_stems(t)
    for rx, repl in SQUASH_FIXES:
        t = rx.sub(repl, t)
    if is_title:
        for rx in TITLE_TEMPLATE_RES.values():
            t = rx.sub("", t)
    else:
        for rx in BOILERPLATE_RES.values():
            t = rx.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

# --- Laporan: berapa baris yang terkena tiap pola (untuk verifikasi manual) ---
rep = []
for name, rx in TITLE_TEMPLATE_RES.items():
    rep.append(("judul", name, int(news["title"].fillna("").str.contains(rx).sum())))
for name, rx in BOILERPLATE_RES.items():
    rep.append(("full_text", name, int(news["full_text"].fillna("").str.contains(rx).sum())))
rep.append(("judul", "kata_menempel_geopolitic", int(news["title"].fillna("").str.contains(r"(?i)[a-z]geopolitic|geopolitic(?:al(?!ly)|s)[a-z]").sum())))
print(pd.DataFrame(rep, columns=["kolom", "pola", "baris_terkena"]).to_string(index=False))

title_before = news["title"].copy()
news["title"]           = news["title"].apply(lambda t: clean_raw_text(t, is_title=True))
news["preview_summary"] = news["preview_summary"].apply(lambda t: clean_raw_text(t))
news["full_text"]       = news["full_text"].apply(lambda t: clean_raw_text(t))

changed = news["title"] != title_before
print(f"\nJudul yang berubah: {int(changed.sum())} dari {len(news)}")
for b, a in list(zip(title_before[changed], news.loc[changed, "title"]))[:6]:
    print(f"  - {b}\n    -> {a}")

# --- Diagnostik: token panjang yang bukan kata kamus (kandidat kata menempel yang belum tertangani) ---
def suspicious_tokens(series, min_len=14, top=30):
    cnt = Counter()
    for s in series.dropna():
        cnt.update(set(re.findall(r"[a-z]{%d,}" % min_len, s.lower())))
    rows = [(w, c) for w, c in cnt.items() if not wordnet.synsets(w)]
    return pd.DataFrame(sorted(rows, key=lambda x: -x[1])[:top], columns=["token", "jumlah_artikel"])

print("\nToken panjang non-kamus pada judul (periksa manual):")
print(suspicious_tokens(news["title"]).to_string(index=False))

In [ ]:
# Near-duplicate: artikel hampir identik (sindikasi / revisi artikel yang sama) dalam jendela +-N hari
from sklearn.feature_extraction.text import TfidfVectorizer

def drop_near_duplicates(df, threshold=0.90, window_days=3, chunk=1000):
    df = df.sort_values("date_raw_dt").reset_index(drop=True)
    texts = (df["title"].fillna("") + " " + df["preview_summary"].fillna("")).str.lower()
    X = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True).fit_transform(texts)
    dates = df["date_raw_dt"].values
    drop = np.zeros(len(df), dtype=bool)
    for start in range(0, len(df), chunk):
        S = (X[start:start + chunk] @ X.T).toarray()
        for li in range(S.shape[0]):
            i = start + li
            if drop[i]:
                continue
            cand = np.where(S[li] >= threshold)[0]
            cand = cand[cand > i]
            if cand.size == 0:
                continue
            gap_days = np.abs((dates[cand] - dates[i]).astype("timedelta64[h]").astype(int)) / 24.0
            drop[cand[gap_days <= window_days]] = True
    return df.loc[~drop].reset_index(drop=True), int(drop.sum())

n_before = len(news)
news, n_dropped = drop_near_duplicates(news, NEAR_DUP_THRESHOLD, NEAR_DUP_WINDOW_DAY)
print(f"Near-duplicate dibuang : {n_dropped} (dari {n_before} -> {len(news)})")

## 2. Data News Cleaning

In [ ]:
# Menggabungkan title, preview, dan full_text
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

def drop_leading_preview(full: str, prev: str) -> str:
    """full_text CNBC biasanya diawali ringkasan yang sama dengan preview_summary -> hindari teks ganda."""
    p = prev.strip().rstrip(".\u2026").strip()
    if len(p) >= 30 and full.lower().startswith(p.lower()):
        return full[len(p):].lstrip(" .\u2026")
    return full

news["body_text"] = [drop_leading_preview(f, p) for f, p in zip(news["full_text"], news["preview_summary"])]

# TITLE_REPEAT=1 -> judul sekali saja (sebelumnya digandakan, yang menggandakan bobot judul)
news["combined_raw"] = (
    ((news["title"] + ". ") * TITLE_REPEAT)
    + news["preview_summary"] + " "
    + news["body_text"]
).str.strip()

print(f"Jumlah baris: {len(news)}")
dup_share = (news["full_text"].str.len() - news["body_text"].str.len()).clip(lower=0).sum() / max(news["full_text"].str.len().sum(), 1)
print(f"Porsi teks ganda (preview di awal full_text) yang dibuang: {dup_share:.2%}")
news[["title", "combined_raw"]].head(3)

In [ ]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

In [ ]:
# Cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")
MULTI_SPACE_RE = re.compile(r"\s+")


def light_clean(text: str) -> str:
    """Pembersihan RINGAN untuk VADER / transformer: huruf besar & tanda baca dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = ZERO_WIDTH_RE.sub("", t)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    return MULTI_SPACE_RE.sub(" ", t).strip()


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan.
    Perbaikan: kontraksi negasi dipertahankan (wouldn't -> would not), U.S. -> US."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = ZERO_WIDTH_RE.sub("", t)
    t = t.replace("\u2019", "'").replace("\u2018", "'")
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = re.sub(r"\bU\.S\b\.?", "US", t)
    t = re.sub(r"\bU\.K\b\.?", "UK", t)
    t = re.sub(r"\bE\.U\b\.?", "EU", t)
    t = re.sub(r"(?i)\bcan't\b", "cannot", t)
    t = re.sub(r"(?i)\bwon't\b", "will not", t)
    t = re.sub(r"(?i)n't\b", " not", t)
    t = re.sub(r"'s\b", "", t)           # possessive
    t = t.replace("'", "")
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["text_model"] = news["combined_raw"].apply(light_clean)       # untuk VADER / IndoBERT / RoBERTa
news["normalized_text"] = news["combined_raw"].apply(normalize_text)  # untuk TF-IDF / LDA / analisis kata

news[["combined_raw", "normalized_text"]].head(3)

In [ ]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

In [ ]:
# Memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text[:1500])      # 1500 karakter pertama cukup dan jauh lebih cepat
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

## 3. Cleaning Data Kurs

In [ ]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

In [ ]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

# Preprocessing

In [ ]:
# Stopword removal
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "also", "said", "says", "say",
    "would", "could", "one", "two", "get", "got",
}

# Stopword NLTK yang sengaja DIPERTAHANKAN karena membawa makna arah/negasi
# (penting untuk berita kurs: 'up', 'down', 'above', 'below', 'not', ...).
KEEP_WORDS = {"no", "not", "nor", "up", "down", "above", "below", "over", "under", "against", "off"}

# Stopword domain berita: hari, bulan, kata atribusi, dan kata generik yang tidak membeda-bedakan isi.
DOMAIN_STOPWORDS = {
    # hari & bulan
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "january", "february", "march", "april", "may", "june", "july", "august",
    "september", "october", "november", "december",
    # atribusi berita
    "added", "adds", "add", "told", "tell", "tells", "noted", "notes", "according",
    "saying", "called", "including", "include", "includes",
    "reuters", "afp", "bloomberg", "getty", "images", "photo", "contributed",
    "subscribe", "newsletter", "inbox", "update", "updates",
    # kata generik
    "even", "still", "many", "much", "well", "part", "around", "far", "another",
    "make", "made", "see", "seen", "go", "going", "new", "first", "last", "next",
    "year", "years", "week", "weeks", "month", "months", "day", "days", "time",
}

STOPWORDS_EN = (set(stopwords.words("english")) - KEEP_WORDS) | CUSTOM_STOPWORDS | DOMAIN_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi; buang stopword, token 1 karakter, dan token angka murni.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_EN and len(tok) > 1 and not tok.isdigit()]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

In [ ]:
# Lemmatization (POS aware) + penyaringan jenis kata
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}

# WordNet mengubah 'us' -> 'u' (terlihat pada hasil sebelumnya: 'white house u iran'). Lindungi singkatan ini.
PROTECT_LEMMA = {"us", "uk", "eu"}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_with_pos(tokens: list):
    # Lemmatisasi tiap token berdasarkan POS tag-nya; kembalikan (lemma, tag).
    if not tokens:
        return [], []
    tagged = pos_tag(tokens)
    lemmas = [
        tok if tok in PROTECT_LEMMA else lemmatizer.lemmatize(tok, pos=get_wordnet_pos(tag))
        for tok, tag in tagged
    ]
    return lemmas, [tag for _, tag in tagged]


def lemmatize_tokens(tokens: list) -> list:
    # Dipertahankan agar sel visualisasi di bawah tetap berjalan.
    return lemmatize_with_pos(tokens)[0]


_res = news["tokens_no_stopwords"].apply(lemmatize_with_pos)
news["tokens_lemmatized"] = _res.apply(lambda r: r[0])
_tags = _res.apply(lambda r: r[1])
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

# Kata isi: noun / adjective / VERB. Verb sengaja dipertahankan karena kata gerak harga
# (rise, fall, surge, drop, cut, hike) justru pembawa sinyal arah. Adverb dan sisanya dibuang.
news["tokens_content"] = [
    [lem for lem, tag in zip(L, T) if tag[0] in ("N", "J", "V")]
    for L, T in zip(news["tokens_lemmatized"], _tags)
]
del _res, _tags

news[["text_no_stopwords", "text_lemmatized"]].head(3)

## Filter Kosakata Berdasarkan Document Frequency
- **Terlalu langka** (muncul di < `MIN_DOC_FREQ` artikel): kata acak, typo, nama sekali pakai.
- **Terlalu umum** (muncul di > `MAX_DOC_FREQ_RATIO` artikel): template atau kata hasil seleksi scraper (mis. `geopolitical`, `risk`). Batas sengaja longgar (80%): uji sintetis menunjukkan batas 50% ikut membuang kata pembeda yang sah.

Catatan: perhitungan ini tidak memakai label kurs, jadi tidak membocorkan informasi target.

In [ ]:
def build_df_filter(token_lists, min_df, max_ratio):
    n_docs = len(token_lists)
    df_count = Counter(t for toks in token_lists for t in set(toks))
    keep = {t for t, c in df_count.items() if c >= min_df and c / n_docs <= max_ratio}
    return keep, df_count

VOCAB_KEEP, DF_COUNT = build_df_filter(news["tokens_content"], MIN_DOC_FREQ, MAX_DOC_FREQ_RATIO)

news["tokens_filtered"] = news["tokens_content"].apply(lambda toks: [t for t in toks if t in VOCAB_KEEP])
news["text_filtered"] = news["tokens_filtered"].apply(" ".join)

n_docs = len(news)
too_common = sorted([(t, c) for t, c in DF_COUNT.items() if c / n_docs > MAX_DOC_FREQ_RATIO], key=lambda x: -x[1])
too_rare = sum(1 for c in DF_COUNT.values() if c < MIN_DOC_FREQ)
print(f"Kosakata awal            : {len(DF_COUNT)}")
print(f"Dibuang (terlalu langka) : {too_rare}")
print(f"Dibuang (terlalu umum)   : {len(too_common)}  -> {[t for t, _ in too_common[:25]]}")
print(f"Kosakata akhir           : {len(VOCAB_KEEP)}")
print(f"Rata-rata token/artikel  : {news['tokens_filtered'].apply(len).mean():.1f}")
news[["text_lemmatized", "text_filtered"]].head(3)

In [ ]:
news

# Penyelerasan Data

In [ ]:
# Ubah timestamp publikasi ke WIB
# Kolom mana yang valid? Coba beberapa kandidat dan pilih yang parse-rate-nya tertinggi.
candidates = [c for c in ["published_iso", "published_raw"] if c in news.columns]
parse_rate = {}
parsed = {}
for c in candidates:
    parsed[c] = pd.to_datetime(news[c], errors="coerce", utc=True)
    parse_rate[c] = parsed[c].notna().mean()
print("Parse-rate kolom waktu publikasi:", {k: f"{v:.1%}" for k, v in parse_rate.items()})

PUB_COL = max(parse_rate, key=parse_rate.get) if parse_rate else None
if PUB_COL is not None and parse_rate[PUB_COL] > 0:
    news["published_wib"] = parsed[PUB_COL].dt.tz_convert("Asia/Jakarta")
else:
    news["published_wib"] = pd.NaT
print(f"Kolom dipakai: {PUB_COL}")

# Verifikasi zona waktu: bandingkan dengan date_raw_dt (naive). Selisih konstan (mis. 11/12 jam)
# berarti date_raw_dt BUKAN WIB dan tidak boleh dipakai untuk memotong jam 15:00 WIB.
if news["published_wib"].notna().any():
    diff_h = (news["published_wib"].dt.tz_localize(None) - news["date_raw_dt"]).dt.total_seconds().div(3600).round()
    print("\nSelisih (published_wib - date_raw_dt) dalam jam, 6 nilai terbanyak:")
    print(diff_h.value_counts().head(6).to_string())

# Waktu acuan untuk penyelarasan: utamakan WIB; fallback ke date_raw_dt (dan ditandai).
event_time = news["published_wib"].dt.tz_localize(None)
news["time_source"] = np.where(event_time.notna(), "published_wib", "date_raw_dt_fallback")
news["event_time"] = event_time.fillna(news["date_raw_dt"])
print("\nSumber waktu:", news["time_source"].value_counts().to_dict())

news["date"] = news["event_time"].dt.date
news["time"] = news["event_time"].dt.time

In [ ]:
# Penyelarasan ke hari bursa BI.
# PERBAIKAN:
#  - memakai waktu WIB (event_time), bukan date_raw_dt mentah
#  - hari bursa diambil dari kalender kurs BI itu sendiri, sehingga LIBUR NASIONAL ikut ditangani
#    (versi lama hanya menggeser akhir pekan; berita yang jatuh di hari libur hilang saat merge)
BI_DATES = pd.DatetimeIndex(sorted(pd.to_datetime(bi["Tanggal_dt"]).dt.normalize().unique()))

def align_to_bi_dates(ts: pd.Series, cutoff_hour: int = CUTOFF_HOUR_WIB) -> pd.Series:
    day = ts.dt.normalize()
    after_cutoff = ts > (day + pd.Timedelta(hours=cutoff_hour))
    target = day + pd.to_timedelta(after_cutoff.astype(int), unit="D")   # geser +1 hari jika lewat cutoff
    vals = target.values.astype("datetime64[ns]")
    pos = BI_DATES.values.searchsorted(vals)                            # hari BI pertama >= target (NaT -> len)
    ok = target.notna().values & (pos < len(BI_DATES))
    out = np.full(len(ts), np.datetime64("NaT"), dtype="datetime64[ns]")
    out[ok] = BI_DATES.values[pos[ok]]
    return pd.Series(pd.to_datetime(out), index=ts.index)

news["aligned_trading_date"] = align_to_bi_dates(news["event_time"])

# Diagnostik: berapa artikel yang jatuh di hari libur BI (tidak ada di kalender kurs) dan digeser
_day = news["event_time"].dt.normalize() + pd.to_timedelta((news["event_time"] > news["event_time"].dt.normalize() + pd.Timedelta(hours=CUTOFF_HOUR_WIB)).astype(int), unit="D")
_wk = _day.dt.weekday
_naive = _day + pd.to_timedelta((_wk == 5).astype(int) * 2 + (_wk == 6).astype(int), unit="D")
print(f"Artikel yang jatuh di hari libur (bukan akhir pekan) dan digeser ke hari bursa berikutnya: {(~_naive.isin(BI_DATES)).sum()}")
print(f"Artikel tanpa tanggal bursa (di luar rentang kurs): {news['aligned_trading_date'].isna().sum()}")
news = news.dropna(subset=["aligned_trading_date"]).reset_index(drop=True)

# =========================================================================
# EXPORT CSV: DATA BERITA SEBELUM LEMMATIZE DKK (TAPI SUDAH ALIGNED)
# =========================================================================
os.makedirs('data/processed', exist_ok=True)

cols_unprocessed = [
    c for c in ['url', 'title', 'preview_summary', 'full_text', 'combined_raw', 'text_model',
                'normalized_text', 'lower_text', 'keyword_matched', 'date_raw_dt', 'event_time',
                'time_source', 'aligned_trading_date']
    if c in news.columns
]
news_aligned_unprocessed = news[cols_unprocessed].copy()

unprocessed_csv_path = 'data/processed/news_aligned_unprocessed.csv'
news_aligned_unprocessed.to_csv(unprocessed_csv_path, index=False)

print("=" * 75)
print("BERHASIL MENYIMPAN CSV (BELUM LEMMATIZE DKK, TAPI SUDAH ALIGNED):")
print(f"Path   : {unprocessed_csv_path}")
print(f"Ukuran : {news_aligned_unprocessed.shape[0]} baris x {news_aligned_unprocessed.shape[1]} kolom")
print(f"Kolom  : {list(news_aligned_unprocessed.columns)}")
print("=" * 75)
news_aligned_unprocessed.head(3)

# VADER

In [ ]:
# Agregasi harian per tanggal bursa BI
news_sorted = news.sort_values('event_time')

# Kolom bertipe list (token) tidak ikut diagregasi (sebelumnya di-str()-kan jadi teks raksasa)
LIST_COLS = [c for c in news_sorted.columns if news_sorted[c].map(lambda v: isinstance(v, list)).any()]

text_cols = [
    c for c in news_sorted.select_dtypes(include=['object', 'string']).columns
    if c not in ['aligned_trading_date'] + LIST_COLS
]

def join_pipe(x):
    return ' | '.join(x.fillna('').astype(str))

agg_spec = {'news_count': ('title', 'size')}
for c in text_cols:
    agg_spec[f'all_{c}'] = (c, join_pipe)

daily_news = (
    news_sorted.groupby('aligned_trading_date')
               .agg(**agg_spec)
               .reset_index()
)
print(f"Kolom list dilewati: {LIST_COLS}")
daily_news

In [ ]:
# PERBAIKAN: merge memakai `bi` (hasil cleaning: rentang tanggal, numerik valid, Jual > Beli, dedup),
# bukan `bi_raw`. Sebelumnya hasil cleaning kurs dibuang begitu saja.
bi["Tanggal_dt"] = pd.to_datetime(bi["Tanggal_dt"]).dt.normalize()
daily_news["aligned_trading_date"] = pd.to_datetime(daily_news["aligned_trading_date"]).dt.normalize()

print("Duplikat kunci BI   :", bi["Tanggal_dt"].duplicated().sum())
print("Duplikat kunci berita:", daily_news["aligned_trading_date"].duplicated().sum())
orphan = set(daily_news["aligned_trading_date"]) - set(bi["Tanggal_dt"])
print("Tanggal berita yang tidak punya kurs (harus 0):", len(orphan))

merged_data = pd.merge(
    bi,
    daily_news,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
    validate='one_to_one'
).drop(columns='aligned_trading_date')

# Hari tanpa berita: isi supaya tidak NaN
merged_data['news_count'] = merged_data['news_count'].fillna(0).astype(int)
text_cols_m = [c for c in merged_data.columns if c.startswith('all_')]
merged_data[text_cols_m] = merged_data[text_cols_m].fillna('')

print(f"Hari bursa: {len(merged_data)} | hari berberita: {(merged_data['news_count'] > 0).sum()}")
merged_data

## Label Arah Kurs
Dua definisi dibuat eksplisit agar tidak tertukar:

| Kolom | Arti | Cocok untuk |
|---|---|---|
| `y_same_day` | kurs hari *d* vs hari *d-1* (1 = naik / rupiah melemah) | analisis asosiasi kata-arah (seperti sel analisis di bawah) |
| `y_next_day` | kurs hari *d+1* vs hari *d* | **prediksi** (berita hingga jam cutoff hari *d* meramal gerak berikutnya) |

Jika berita hari *d* sudah ikut bereaksi terhadap gerak kurs hari *d*, `y_same_day` akan tampak "memprediksi" padahal hanya mengikuti. Hari dengan perubahan tepat 0 diberi `NaN`.

In [ ]:
merged_data = merged_data.sort_values("Tanggal_dt").reset_index(drop=True)
merged_data["rate_mid"] = (merged_data["Kurs Jual"] + merged_data["Kurs Beli"]) / 2.0

merged_data["ret_same_day"] = merged_data["rate_mid"].pct_change()
merged_data["ret_next_day"] = merged_data["rate_mid"].shift(-1) / merged_data["rate_mid"] - 1.0

def _direction(ret):
    return np.where(ret > 0, 1.0, np.where(ret < 0, 0.0, np.nan))

merged_data["y_same_day"] = _direction(merged_data["ret_same_day"])
merged_data["y_next_day"] = _direction(merged_data["ret_next_day"])
merged_data.loc[merged_data["ret_same_day"].isna(), "y_same_day"] = np.nan
merged_data.loc[merged_data["ret_next_day"].isna(), "y_next_day"] = np.nan

for col in ["y_same_day", "y_next_day"]:
    vc = merged_data[col].value_counts(dropna=False).to_dict()
    print(f"{col}: {vc}")

In [ ]:
# drop column
COL = [
    "Nilai", "Tanggal", "all_url", "all_section", "all_published_date", "all_published_time",
    "all_published_iso", "all_scrape_status", "all_tanggal", "all_jam", "all_detected_lang",
    "all_date", "all_time",
    # kolom bantu baru
    "all_url_clean", "all_title_norm", "all_time_source", "all_body_text",
]

merged_data_clean = merged_data.drop(columns=COL, errors="ignore")
merged_data_clean

In [ ]:
# =========================================================================
# EXPORT CSV: DATA HASIL PREPROCESSING LENGKAP
# =========================================================================
os.makedirs('data/processed', exist_ok=True)

# 1. Artikel yang SUDAH DIPROSES tapi TIDAK DIMERGE (level artikel)
cols_unmerged = [
    c for c in ['url', 'title', 'preview_summary', 'full_text', 'combined_raw', 'text_model',
                'normalized_text', 'lower_text', 'text_no_stopwords', 'text_lemmatized', 'text_filtered',
                'keyword_matched', 'date_raw_dt', 'event_time', 'time_source', 'tanggal', 'jam',
                'aligned_trading_date']
    if c in news.columns
]
news_processed_unmerged = news[cols_unmerged].copy()
unmerged_csv_path = 'data/processed/news_processed_unmerged.csv'
news_processed_unmerged.to_csv(unmerged_csv_path, index=False)

# 2. Final clean (sudah diproses + dimerge kurs BI harian + label arah)
clean_csv_path = 'data/processed/merged_data_clean.csv'
merged_data_clean.to_csv(clean_csv_path, index=False)

# 3. Daily merged khusus teks sebelum lemmatize
_processed_daily = ['all_tokens_no_stopwords', 'all_text_no_stopwords', 'all_tokens_lemmatized',
                    'all_text_lemmatized', 'all_text_filtered', 'all_tokens_filtered', 'all_tokens_content']
cols_unprocessed_daily = [c for c in merged_data_clean.columns if c not in _processed_daily]
daily_unprocessed_path = 'data/processed/merged_data_unprocessed.csv'
merged_data_clean[cols_unprocessed_daily].to_csv(daily_unprocessed_path, index=False)

print("=" * 75)
print("BERHASIL MENYIMPAN OUTPUT CSV:")
print(f"1. Artikel Diproses (TIDAK DIMERGE)       : {unmerged_csv_path}")
print(f"   Ukuran : {news_processed_unmerged.shape[0]} baris x {news_processed_unmerged.shape[1]} kolom")
print(f"2. Final Clean (Lemmatize + Dimerge + Label): {clean_csv_path}")
print(f"   Ukuran : {merged_data_clean.shape[0]} baris x {merged_data_clean.shape[1]} kolom")
print(f"3. Daily Merged (Belum Lemmatize + Dimerge): {daily_unprocessed_path}")
print(f"   Ukuran : {merged_data_clean[cols_unprocessed_daily].shape[0]} baris x {len(cols_unprocessed_daily)} kolom")
print("=" * 75)
print("\nPreview 3 baris teratas news_processed_unmerged:")
display(news_processed_unmerged.head(3))

# Visualisasi Word Cloud (Judul vs Teks Lengkap)
Visualisasi perbandingan distribusi kata kunci penting antara **Judul (Headline)** dan **Teks Lengkap (Full-Text)** setelah pembersihan tanda baca dan stopword removal.

In [ ]:
!pip install wordcloud

In [ ]:
# Visualisasi Word Cloud untuk Judul dan Teks Lengkap setelah Stopword Removal
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from pathlib import Path
import html
import re

# Helper pembersih & filter kata setelah stopword removal
def get_clean_words(series):
    cleaned_words = []
    for text in series.dropna():
        t = html.unescape(str(text).lower())
        t = re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", t)
        words = [w for w in t.split() if w not in STOPWORDS_EN and len(w) > 2]
        cleaned_words.extend(words)
    return " ".join(cleaned_words)

print("Membuat teks gabungan untuk Word Cloud...")
title_words = get_clean_words(news["title"])
fulltext_words = get_clean_words(news["full_text"])

# 1. Word Cloud untuk Judul (Title)
wc_title = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="Blues",
    max_words=120,
    random_state=42
).generate(title_words)

# 2. Word Cloud untuk Teks Lengkap (Full-Text)
wc_fulltext = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="YlOrRd",
    max_words=120,
    random_state=42
).generate(fulltext_words)

# 3. Tampilkan Visualisasi Berdampingan
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_title, interpolation="bilinear")
axes[0].set_title("Word Cloud - Judul Berita (Title)\n(Setelah Stopword Removal)", fontsize=15, fontweight="bold", pad=15)
axes[0].axis("off")

axes[1].imshow(wc_fulltext, interpolation="bilinear")
axes[1].set_title("Word Cloud - Teks Lengkap (Full-Text)\n(Setelah Stopword Removal)", fontsize=15, fontweight="bold", pad=15)
axes[1].axis("off")

plt.tight_layout(pad=2.0)

# Simpan gambar untuk kebutuhan laporan
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
output_img_path = figures_dir / "wordcloud_comparison.png"
fig.savefig(output_img_path, dpi=300, bbox_inches="tight")
print(f"Visualisasi berhasil disimpan ke: {output_img_path}")

plt.show()


# Visualisasi Word Cloud Setelah Process Full (Lemmatized)
Visualisasi Word Cloud untuk **Judul** dan **Teks Lengkap** setelah seluruh proses NLP (*stopword removal* dan *POS-aware lemmatization*) selesai. Kata-kata jamak dan berimbuhan telah dikembalikan ke bentuk kata dasarnya (misal: *prices* -> *price*, *markets* -> *market*).

In [ ]:
# Visualisasi Word Cloud Setelah Process Full (Cleaning + Stopwords + Lemmatization)
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from pathlib import Path
import html
import re

print("Membuat Word Cloud setelah process full (lemmatized)...")
title_token_docs = [
    [w for w in re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", html.unescape(str(t).lower())).split() if w not in STOPWORDS_EN and len(w) > 2]
    for t in news["title"].dropna()
]
title_lem_words = [w for doc in title_token_docs for w in lemmatize_tokens(doc)]

# Ambil teks lengkap yang sudah dilemmatisasi di notebook
if "text_lemmatized" in news.columns:
    fulltext_lem_text = " ".join(news["text_lemmatized"].dropna())
else:
    fulltext_lem_text = " ".join([lemmatizer.lemmatize(w) for w in fulltext_words.split()])

# 1. Word Cloud Judul (Full Process)
wc_title_full = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="Blues",
    max_words=120,
    random_state=42
).generate(" ".join(title_lem_words))

# 2. Word Cloud Teks Lengkap (Full Process)
wc_fulltext_full = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="YlOrRd",
    max_words=120,
    random_state=42
).generate(fulltext_lem_text)

# 3. Visualisasi Berdampingan
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_title_full, interpolation="bilinear")
axes[0].set_title("Word Cloud: Judul Berita\n(Setelah Process Full / Lemmatized)", fontsize=15, fontweight="bold", pad=15)
axes[0].axis("off")

axes[1].imshow(wc_fulltext_full, interpolation="bilinear")
axes[1].set_title("Word Cloud: Teks Lengkap\n(Setelah Process Full / Lemmatized)", fontsize=15, fontweight="bold", pad=15)
axes[1].axis("off")

plt.tight_layout(pad=2.0)

# Simpan ke folder reports/figures
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
save_path = figures_dir / "wordcloud_full_process.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Gambar Word Cloud (Process Full) berhasil disimpan ke: {save_path}")

plt.show()


# Visualisasi Co-occurrence Network (Jaringan Keterhubungan Kata Kunci)
Visualisasi graf jaringan keterhubungan (**Co-occurrence Network**) antar kata kunci penting pada judul berita geopolitik dan ekonomi. Setiap *node* merepresentasikan kata kunci, *garis/edge* merepresentasikan frekuensi kemunculan bersama dalam satu judul, dan *warna* menunjukkan klaster topik komunitas (*topic cluster*) yang terdeteksi otomatis.

In [ ]:
# Visualisasi Co-occurrence Network untuk Judul Berita
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import html
import re

print("Menganalisis Co-occurrence Network...")

# 1. Ekstraksi kata per judul (setelah stopword removal & lemmatization)
docs = []
for t in news["title"].dropna():
    cleaned = re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", html.unescape(str(t).lower()))
    words = [w for w in cleaned.split() if w not in STOPWORDS_EN and len(w) > 2]
    lem_words = set(lemmatize_tokens(words))
    docs.append(lem_words)

# 2. Ambil top 35 kata teratas dan hitung pasangan ko-okurensi
word_counts = Counter(w for doc in docs for w in doc)
top_words = set([w for w, c in word_counts.most_common(35)])

co_occur = defaultdict(int)
for doc in docs:
    shared = [w for w in doc if w in top_words]
    for w1, w2 in combinations(sorted(shared), 2):
        co_occur[(w1, w2)] += 1

# 3. Bentuk Graph NetworkX
G = nx.Graph()
for word in top_words:
    G.add_node(word, count=word_counts[word])

# Hubungkan pasangan yang muncul bersama minimal 10 kali
min_co_occur = 10
for (w1, w2), weight in co_occur.items():
    if weight >= min_co_occur:
        G.add_edge(w1, w2, weight=weight)

# Bersihkan node tanpa relasi
G.remove_nodes_from(list(nx.isolates(G)))

# 4. Deteksi Klaster Komunitas & Pewarnaan
communities = list(nx.community.greedy_modularity_communities(G))
palette = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]
node_color_map = {}
for i, comm in enumerate(communities):
    color = palette[i % len(palette)]
    for node in comm:
        node_color_map[node] = color

node_colors = [node_color_map[n] for n in G.nodes()]
node_sizes = [word_counts[n] * 6.5 + 400 for n in G.nodes()]
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_widths = [0.8 + (w / max(edge_weights)) * 5.0 for w in edge_weights]

# 5. Layout dan Visualisasi
pos = nx.spring_layout(G, k=0.55, iterations=80, seed=42)
plt.figure(figsize=(16, 11), dpi=200)

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.35, edge_color="#7f8c8d")
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.88, edgecolors="#2c3e50", linewidths=1.8)
nx.draw_networkx_labels(
    G, pos,
    labels={n: n for n in G.nodes()},
    font_size=11, font_family="sans-serif", font_weight="bold", font_color="#1a1a1a",
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.75)
)

plt.title("Co-occurrence Network: Keterhubungan Kata Kunci Berita Geopolitik & Pasar\n(Warna = Klaster Topik, Ukuran Node = Frekuensi, Tebal Garis = Kekuatan Relasi)", fontsize=14, fontweight="bold", pad=25)
plt.axis("off")
plt.tight_layout()

# Simpan gambar otomatis untuk laporan
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
output_path = figures_dir / "co_occurrence_network.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Visualisasi Co-occurrence Network berhasil disimpan ke: {output_path}")

plt.show()


# Analisis Kata Pembeda: Hari Kurs Naik vs Kurs Turun (Teks Mentah Setelah Stopword Removal)
Berbeda dengan *Word Cloud* biasa yang hanya menampilkan kata yang paling sering muncul secara global (seperti *oil*, *china*, *russia*), analisis ini membandingkan **frekuensi kemunculan relatif (%)** dan **Log-Odds Ratio** tiap kata langsung dari **teks mentah setelah *stopword removal* saja** (`all_title` dan `all_full_text`, **tanpa *lemmatization***) antara **hari saat kurs USD/IDR Naik (Rupiah melemah)** vs **hari saat kurs USD/IDR Turun (Rupiah menguat)**.

Kata dengan nilai $\log_2\text{Odds}$ positif tertinggi condong muncul pada hari kurs **NAIK**, sedangkan nilai negatif tertinggi condong muncul pada hari kurs **TURUN** — menunjukkan kata mana yang benar-benar memiliki **daya pembeda untuk klasifikasi arah**.

In [ ]:
# Analisis & Visualisasi Kata Pembeda: Hari Kurs Naik vs Hari Kurs Turun (Teks Mentah + Stopword Removal)
import re
import html
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 1. Gunakan merged_data_clean dari memori jika tersedia, atau muat dari CSV
if "merged_data_clean" in globals():
    df_disc = merged_data_clean.copy()
else:
    csv_cand = [
        Path("data/processed/merged_data_clean.csv"),
        Path("../data/processed/merged_data_clean.csv"),
        Path("percobaan task 3/data/processed/merged_data_clean.csv"),
    ]
    csv_p = next((p for p in csv_cand if p.exists()), None)
    if csv_p is None:
        raise FileNotFoundError("File merged_data_clean.csv tidak ditemukan!")
    df_disc = pd.read_csv(csv_p)

df_disc["Tanggal_dt"] = pd.to_datetime(df_disc["Tanggal_dt"])
df_disc = df_disc.sort_values("Tanggal_dt").reset_index(drop=True)
df_disc["rate_mid"] = (df_disc["Kurs Jual"] + df_disc["Kurs Beli"]) / 2.0
df_disc["ret"] = df_disc["rate_mid"].pct_change()

# Ambil hanya hari bursa yang memiliki berita dan pergerakan kurs (naik/turun)
df_disc = df_disc[(df_disc["news_count"] > 0) & df_disc["ret"].notna() & (df_disc["ret"] != 0)].copy()
df_disc["is_up"] = (df_disc["ret"] > 0).astype(int)  # 1 = Kurs Naik (IDR Melemah), 0 = Kurs Turun (IDR Menguat)

n_up_days = int((df_disc["is_up"] == 1).sum())
n_dn_days = int((df_disc["is_up"] == 0).sum())
print(f"Total hari berberita dianalisis: {len(df_disc)} hari (Naik={n_up_days} hari, Turun={n_dn_days} hari)")

# Gabungkan stopword Bahasa Inggris + filler editorial berita + pecahan kontraksi (tanpa lemmatization)
stop_base = set(STOPWORDS_EN) if "STOPWORDS_EN" in globals() else set(ENGLISH_STOP_WORDS)
STOP_DISC = stop_base | set(ENGLISH_STOP_WORDS) | {
    "cnbc", "said", "says", "say", "article", "read", "also", "would", "could",
    "new", "year", "years", "one", "two", "week", "weeks", "day", "days",
    "month", "months", "time", "percent", "today", "bad", "good", "must", "used",
    "don", "didn", "doesn", "isn", "wasn", "aren", "won", "wouldn", "couldn", "shouldn",
    "haven", "hasn", "hadn"
}

def compute_discriminative_words(df_in, col_name, min_days=10):
    up_cnt, dn_cnt = Counter(), Counter()
    for _, row in df_in.iterrows():
        raw = str(row[col_name])
        if not raw or raw.lower() == "nan":
            continue
        # Pembersihan teks mentah + stopword removal saja (tanpa stemming / lemmatization)
        t = html.unescape(raw.lower()).replace("’", "'").replace("‘", "'")
        t = re.sub(r"http\S+|www\.\S+|<[^>]+>", " ", t)
        t = re.sub(r"'s\b", "", t)
        t = re.sub(r"[^a-z\s]", " ", t)
        toks = [w for w in t.split() if len(w) > 2 and w not in STOP_DISC]

        # Hitung kemunculan unik per hari (Document Frequency) agar 1 artikel panjang tidak mendominasi
        unique_toks = set(toks)
        if row["is_up"] == 1:
            up_cnt.update(unique_toks)
        else:
            dn_cnt.update(unique_toks)

    rows = []
    vocab = set(up_cnt.keys()) | set(dn_cnt.keys())
    for w in vocab:
        cu, cd = up_cnt[w], dn_cnt[w]
        if cu + cd < min_days:
            continue
        # Proporsi kemunculan per kelas & Smoothed Log2-Odds Ratio
        p_up = (cu + 0.5) / (n_up_days + 1.0)
        p_dn = (cd + 0.5) / (n_dn_days + 1.0)
        log2_odds = float(np.log2((p_up / (1.0 - p_up)) / (p_dn / (1.0 - p_dn))))
        pct_up = cu / n_up_days * 100.0
        pct_dn = cd / n_dn_days * 100.0
        rows.append({
            "Kata": w,
            "Hari_Naik": cu,
            "Hari_Turun": cd,
            "Pct_Naik (%)": round(pct_up, 2),
            "Pct_Turun (%)": round(pct_dn, 2),
            "Selisih_Pct (%)": round(pct_up - pct_dn, 2),
            "Log2_Odds": round(log2_odds, 4)
        })
    return pd.DataFrame(rows)

# 2. Analisis pada Teks Mentah setelah Stopword Removal (all_title & all_full_text, tanpa lemmatization)
disc_title = compute_discriminative_words(df_disc, "all_title", min_days=10)
disc_body  = compute_discriminative_words(df_disc, "all_full_text", min_days=40)

print("\n=== TOP 10 KATA PEMBEDA DI JUDUL MENTAH (STOPWORD REMOVAL): CONDONG KURS NAIK (RUPIAH MELEMAH) ===")
display(disc_title.nlargest(10, "Log2_Odds").reset_index(drop=True))

print("\n=== TOP 10 KATA PEMBEDA DI JUDUL MENTAH (STOPWORD REMOVAL): CONDONG KURS TURUN (RUPIAH MENGUAT) ===")
display(disc_title.nsmallest(10, "Log2_Odds").reset_index(drop=True))

print("\n=== TOP 10 KATA PEMBEDA DI FULL-TEXT MENTAH (STOPWORD REMOVAL): CONDONG KURS NAIK ===")
display(disc_body.nlargest(10, "Log2_Odds").reset_index(drop=True))

print("\n=== TOP 10 KATA PEMBEDA DI FULL-TEXT MENTAH (STOPWORD REMOVAL): CONDONG KURS TURUN ===")
display(disc_body.nsmallest(10, "Log2_Odds").reset_index(drop=True))

# 3. Visualisasi Diverging Bar Chart (Judul Mentah vs Full-Text Mentah setelah Stopword Removal)
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), dpi=150)

for ax, df_src, judul_panel in [
    (axes[0], disc_title, "Kata Pembeda pada Judul Mentah (all_title, Stopword Removal)"),
    (axes[1], disc_body,  "Kata Pembeda pada Full-Text Mentah (all_full_text, Stopword Removal)")
]:
    top_up = df_src.nlargest(10, "Log2_Odds")
    top_dn = df_src.nsmallest(10, "Log2_Odds")
    comb = pd.concat([top_dn, top_up]).drop_duplicates("Kata").sort_values("Log2_Odds")

    colors = ["#2a9d8f" if v < 0 else "#e63946" for v in comb["Log2_Odds"]]
    bars = ax.barh(comb["Kata"], comb["Log2_Odds"], color=colors, edgecolor="none", alpha=0.9)
    ax.axvline(0, color="black", linewidth=1.0)
    max_abs = float(comb["Log2_Odds"].abs().max())
    ax.set_xlim(-max_abs * 1.32, max_abs * 1.32)
    ax.set_title(f"{judul_panel}\n(Kiri/Hijau = Condong Kurs Turun | Kanan/Merah = Condong Kurs Naik)", fontweight="bold", fontsize=11)
    ax.set_xlabel("Log2 Odds Ratio (Daya Pembeda Arah Kurs)")
    ax.grid(axis="x", linestyle=":", alpha=0.6)

    # Anotasi frekuensi hari (Naik vs Turun) di samping batang
    for bar, (_, r) in zip(bars, comb.iterrows()):
        val = r["Log2_Odds"]
        txt = f" {r['Hari_Naik']} vs {r['Hari_Turun']} "
        ha = "left" if val >= 0 else "right"
        ax.text(val, bar.get_y() + bar.get_height()/2, txt, va="center", ha=ha, fontsize=8, color="#333333")

plt.tight_layout()
fig_dir = Path("percobaan task 3/reports/figures")
if not fig_dir.parent.parent.exists():
    fig_dir = Path("reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
out_fig = fig_dir / "discriminative_words_up_vs_down.png"
plt.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafik kata pembeda berhasil disimpan ke: {out_fig}")


## Visualisasi Word Cloud Diskriminatif (Teks Mentah Setelah Stopword Removal)
Menggunakan `generate_from_frequencies` berbobot **$|\log_2\text{Odds Ratio}|$** (Top 45 kata pembeda tiap arah dari **teks mentah setelah *stopword removal***), sehingga ukuran kata mencerminkan **daya pembeda** antara hari Kurs Naik vs Kurs Turun — bukan sekadar kata umum yang sering muncul di kedua kondisi.

In [ ]:
# Visualisasi Word Cloud Diskriminatif: Hari Kurs Naik vs Hari Kurs Turun (Teks Mentah + Stopword Removal)
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from wordcloud import WordCloud

# Pangkas ujung warna pucat agar seluruh kata kontras dan tajam di latar putih
cmap_up   = ListedColormap(plt.cm.Reds(np.linspace(0.45, 0.95, 256)))
cmap_down = ListedColormap(plt.cm.YlGn(np.linspace(0.50, 0.95, 256)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=150)

configs = [
    (axes[0, 0], disc_title, "up",   cmap_up,   "Judul Mentah (Stopword Removal) — Condong Kurs NAIK (Rupiah Melemah)"),
    (axes[0, 1], disc_title, "down", cmap_down, "Judul Mentah (Stopword Removal) — Condong Kurs TURUN (Rupiah Menguat)"),
    (axes[1, 0], disc_body,  "up",   cmap_up,   "Full-Text Mentah (Stopword Removal) — Condong Kurs NAIK (Rupiah Melemah)"),
    (axes[1, 1], disc_body,  "down", cmap_down, "Full-Text Mentah (Stopword Removal) — Condong Kurs TURUN (Rupiah Menguat)"),
]

for ax, df_src, direction, cmap, title in configs:
    if direction == "up":
        sub = df_src[df_src["Log2_Odds"] > 0].nlargest(45, "Log2_Odds")
        freqs = dict(zip(sub["Kata"], sub["Log2_Odds"]))
    else:
        sub = df_src[df_src["Log2_Odds"] < 0].nsmallest(45, "Log2_Odds")
        freqs = dict(zip(sub["Kata"], sub["Log2_Odds"].abs()))

    wc = WordCloud(
        width=800,
        height=450,
        background_color="white",
        colormap=cmap,
        max_words=45,
        prefer_horizontal=0.85,
        random_state=42
    ).generate_from_frequencies(freqs)

    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontweight="bold", fontsize=11, pad=8)
    ax.axis("off")

plt.tight_layout()
fig_dir = Path("percobaan task 3/reports/figures")
if not fig_dir.parent.parent.exists():
    fig_dir = Path("reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
out_wc = fig_dir / "wordcloud_discriminative_up_vs_down.png"
plt.savefig(out_wc, dpi=150, bbox_inches="tight")
plt.show()
print(f"Word Cloud diskriminatif berhasil disimpan ke: {out_wc}")


# Seleksi Kata Secara Statistik (hanya data latih)
Word cloud / bar chart log-odds mentah menyorot kata dengan hitungan sangat kecil (mis. `venezuela` 15 vs 1, `april`, `loss` 2 vs 9), yang hampir pasti kebetulan atau perancu waktu. Sel ini memakai prosedur yang lebih ketat:

1. **Split kronologis**: kata dipilih hanya dari periode latih (tidak acak, tidak memakai data uji).
2. **Hitungan per hari** (bukan per token) dengan `min_df` hari minimum.
3. **Log-odds dengan prior informatif** (Monroe dkk., 2008) menghasilkan *z-score* yang menyusutkan kata langka.
4. **Stabilitas waktu**: tanda dan kekuatan efek harus sama di paruh awal dan paruh akhir periode latih.
5. **Uji permutasi per blok**: label diacak per blok hari; kata lolos hanya jika |z| melebihi persentil ke-95 dari z maksimum pada data acak (mengendalikan risiko banyak uji sekaligus).
6. **Cek data uji**: berapa persen kata terpilih yang arahnya sama di periode uji (≈50% berarti hanya kebetulan).

Jika tidak ada kata yang lolos, itu hasil yang sah: artinya tidak ada sinyal kata tunggal yang andal.

In [ ]:
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import CountVectorizer

LABEL_COL  = "y_next_day"          # ganti "y_same_day" untuk analisis asosiasi
TEXT_COL   = "all_text_filtered"   # atau "all_title" / "all_text_lemmatized"
TRAIN_FRAC = 0.70
MIN_DAYS   = 20                    # kata harus muncul di >= 20 hari pada data latih
Z_MIN      = 1.96
N_PERM     = 200
BLOCK      = 20                    # panjang blok hari saat mengacak label
PRIOR_A0   = 500
rng = np.random.default_rng(42)

df_w = merged_data_clean.copy()
df_w = df_w[(df_w["news_count"] > 0) & df_w[LABEL_COL].notna()].sort_values("Tanggal_dt").reset_index(drop=True)
split = int(len(df_w) * TRAIN_FRAC)
train, test = df_w.iloc[:split], df_w.iloc[split:]
y_tr, y_te = train[LABEL_COL].values.astype(int), test[LABEL_COL].values.astype(int)
print(f"Hari latih: {len(train)} ({train['Tanggal_dt'].min().date()} - {train['Tanggal_dt'].max().date()}) | "
      f"hari uji: {len(test)} ({test['Tanggal_dt'].min().date()} - {test['Tanggal_dt'].max().date()})")
print(f"Proporsi naik (latih): {y_tr.mean():.2%} | (uji): {y_te.mean():.2%}")

cv = CountVectorizer(binary=True, min_df=MIN_DAYS, token_pattern=r"(?u)\b[a-z][a-z]+\b")
X_tr = cv.fit_transform(train[TEXT_COL]).tocsr()
X_te = cv.transform(test[TEXT_COL]).tocsr()
vocab = cv.get_feature_names_out()
print(f"Kosakata (>= {MIN_DAYS} hari latih): {len(vocab)}")


def counts_by_class(X, y):
    f1 = np.asarray(X.T @ y).ravel().astype(float)
    f0 = np.asarray(X.sum(axis=0)).ravel().astype(float) - f1
    return f1, f0


def log_odds_z(f1, f0, a0=PRIOR_A0):
    """Log-odds dengan prior Dirichlet informatif (Monroe dkk., 2008); mengembalikan z-score."""
    pooled = f1 + f0
    alpha = pooled / max(pooled.sum(), 1.0) * a0
    n1, n0 = f1.sum(), f0.sum()
    d = (np.log((f1 + alpha) / (n1 + a0 - f1 - alpha)) - np.log((f0 + alpha) / (n0 + a0 - f0 - alpha)))
    var = 1.0 / (f1 + alpha) + 1.0 / (f0 + alpha)
    return d / np.sqrt(var)


# 1) z-score pada seluruh data latih
f1, f0 = counts_by_class(X_tr, y_tr)
z = log_odds_z(f1, f0)

# 2) stabilitas: paruh awal vs paruh akhir data latih
h = len(train) // 2
zA = log_odds_z(*counts_by_class(X_tr[:h], y_tr[:h]))
zB = log_odds_z(*counts_by_class(X_tr[h:], y_tr[h:]))
rho = spearmanr(zA, zB).correlation
print(f"\nKorelasi peringkat z antar paruh periode latih: {rho:.3f}  (≈0 = tidak stabil)")
stable = (np.sign(zA) == np.sign(zB)) & (np.abs(zA) > 1.0) & (np.abs(zB) > 1.0)


# 3) uji permutasi per blok -> ambang z maksimum di bawah hipotesis nol
def block_shuffle(y, block=BLOCK):
    blocks = [y[i:i + block] for i in range(0, len(y), block)]
    order = rng.permutation(len(blocks))
    return np.concatenate([blocks[i] for i in order])

null_max = np.array([np.abs(log_odds_z(*counts_by_class(X_tr, block_shuffle(y_tr)))).max() for _ in range(N_PERM)])
z_perm = float(np.percentile(null_max, 95))
print(f"Ambang permutasi (persentil 95 dari |z| maks acak): {z_perm:.2f}")

sig = np.abs(z) > max(z_perm, Z_MIN)
selected = stable & sig
print(f"Kata lolos z>{Z_MIN} saja: {(np.abs(z) > Z_MIN).sum()} | + stabil: {((np.abs(z) > Z_MIN) & stable).sum()} | + permutasi: {selected.sum()}")

# 4) cek data uji
if selected.any():
    f1_te, f0_te = counts_by_class(X_te, y_te)
    z_te = log_odds_z(f1_te, f0_te)
    has_te = (f1_te + f0_te) > 0
    chk = selected & has_te
    agree = float(np.mean(np.sign(z[chk]) == np.sign(z_te[chk]))) if chk.any() else float("nan")
    print(f"Kesesuaian arah kata terpilih di data uji: {agree:.1%} dari {int(chk.sum())} kata (acak ≈ 50%)")

res = pd.DataFrame({"kata": vocab, "z_latih": z, "z_paruh1": zA, "z_paruh2": zB,
                    "hari_naik": f1.astype(int), "hari_turun": f0.astype(int),
                    "stabil": stable, "lolos_permutasi": sig, "terpilih": selected})
selected_df = res[res["terpilih"]].sort_values("z_latih")
os.makedirs("data/processed", exist_ok=True)
res.sort_values("z_latih").to_csv("data/processed/word_stats_train_only.csv", index=False)
selected_df.to_csv("data/processed/selected_words_train_only.csv", index=False)

if selected_df.empty:
    print("\nTidak ada kata yang lolos seluruh saringan -> tidak ada sinyal kata tunggal yang andal pada data ini.")
    print("Kandidat terkuat (BELUM lolos uji, jangan dijadikan bukti):")
    display(res.reindex(res["z_latih"].abs().sort_values(ascending=False).index).head(15)[["kata", "z_latih", "z_paruh1", "z_paruh2", "hari_naik", "hari_turun"]])
else:
    print(f"\n{len(selected_df)} kata terpilih:")
    display(selected_df[["kata", "z_latih", "z_paruh1", "z_paruh2", "hari_naik", "hari_turun"]])
    fig, ax = plt.subplots(figsize=(8, max(3, 0.28 * len(selected_df))))
    ax.barh(selected_df["kata"], selected_df["z_latih"],
            color=["#e63946" if v > 0 else "#2a9d8f" for v in selected_df["z_latih"]])
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel(f"z-score log-odds ({LABEL_COL}); kanan = condong kurs naik")
    ax.set_title("Kata yang lolos stabilitas waktu + uji permutasi (data latih)", fontweight="bold")
    plt.tight_layout()
    Path("reports/figures").mkdir(parents=True, exist_ok=True)
    plt.savefig("reports/figures/selected_words_train_only.png", dpi=150, bbox_inches="tight")
    plt.show()